In [2]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-sonnet-4-6"

In [3]:
# Helper functions
from anthropic.types import Message

# Magic string to trigger redacted thinking
thinking_test_str = "ANTHROPIC_MAGIC_STRING_TRIGGER_REDACTED_THINKING_46C9A13E193C177646C7398A98432ECCCE4C1253D5E2D82641AC0E52CC2876CB"


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(
    messages,
    system=None,
    temperature=1.0,
    stop_sequences=[],
    tools=None,
    thinking=False,
    thinking_budget=1024,
):
    params = {
        "model": model,
        "max_tokens": 4000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if thinking:
        params["thinking"] = {
            "type": "enabled",
            "budget_tokens": thinking_budget,
        }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])

In [10]:
messages = []

add_user_message(messages, "帮我写一个感人的散文诗")

response= chat(messages, thinking=True)
print(response)


Message(id='msg_01BV9MvMcQY2vX3npMmnZfoM', container=None, content=[ThinkingBlock(signature='EoMCCmUIDRgCKkCK9sF5FUiaBYzFc83K65VvLpH+xkLpePjLpQti8Mu15xC/4CV95bPLWzkBNzJ+ANXd+763orSi/BxeA+kYz+bTMhFjbGF1ZGUtc29ubmV0LTQtNjgAQgh0aGlua2luZxIMFNfeHjWbGw4siI8VGgwplBnZ3gyBUSfSuQMiMHzsVxKALsbAkyPxPzkAfVHtB0R08G9B8vtCSSojg9yQNYZg+2l0SwMeZJbm9PkjkypMm8v8OOpH2sWugH9M/7wvJGDenpgHff+byqo2/tXzBwnnOcL79XciFy+ED4gv12Z/okp/sleayxS68/52gbuEy8tGYHLRDeOijVWx6xgB', thinking='The user wants me to write a touching prose poem in Chinese.', type='thinking'), TextBlock(citations=None, text='# 《你走后，灯还亮着》\n\n---\n\n你走的那天，窗外下着细雨。\n\n不是倾盆的那种，就是那种淅淅沥沥、说不清楚的雨。像一句话说了一半，剩下的藏在喉咙里，再也咽不下去，也再也说不出口。\n\n我站在门口送你。你回头看了我一眼，笑了笑，说"进去吧，外面冷。"\n\n我就真的进去了。\n\n后来我想，我应该再多看你一眼的。应该多站一会儿，哪怕雨把我淋透，哪怕什么话都不说，就那样看着你走远，看到你的背影彻底消失在那条老街的转角处。\n\n可我没有。我就那样关上了门。\n\n---\n\n你喜欢在深夜喝茶。不是什么名贵的茶，就是超市里几块钱一包的茉莉花茶。你说香味淡一点，才不会抢了夜晚的安静。\n\n现在每次我路过茶叶铺，那股淡淡的香气一漫出来，我就不得不停下脚步。\n\n不是伤心。就是有那么一瞬间，时间突然变得很慢很慢，慢到我以为你还在某个地方，正捧着一杯热茶，等我回去。\n\n---\n\n我后来搬了家，换了城市，换了天气，换了所有能换的东

In [11]:
print(response)
print(response.content[0].thinking)
print(response.content[0].thinking)# 遍历 response 的每个 block，按类型分别处理
for i, block in enumerate(response.content):
    print(f"--- Block {i}: {block.type} ---")
    
    if block.type == "thinking":
        # 思考块：内部推理过程
        print(f"[思考内容]\n{block.thinking}")
        print(f"[签名]: {block.signature[:50]}...")  # 签名很长，截取展示
    
    elif block.type == "redacted_thinking":
        # 被屏蔽的思考块（出于安全原因被加密，无法读取明文）
        print(f"[被屏蔽的思考] data: {block.data[:50]}...")
    
    elif block.type == "text":
        # 文本回复块
        print(f"[文本回复]\n{block.text}")
    
    elif block.type == "tool_use":
        # 工具调用块
        print(f"[工具调用] name={block.name}, input={block.input}")
    
    else:
        print(f"[未知类型] {block}")
    
    print()


Message(id='msg_01BV9MvMcQY2vX3npMmnZfoM', container=None, content=[ThinkingBlock(signature='EoMCCmUIDRgCKkCK9sF5FUiaBYzFc83K65VvLpH+xkLpePjLpQti8Mu15xC/4CV95bPLWzkBNzJ+ANXd+763orSi/BxeA+kYz+bTMhFjbGF1ZGUtc29ubmV0LTQtNjgAQgh0aGlua2luZxIMFNfeHjWbGw4siI8VGgwplBnZ3gyBUSfSuQMiMHzsVxKALsbAkyPxPzkAfVHtB0R08G9B8vtCSSojg9yQNYZg+2l0SwMeZJbm9PkjkypMm8v8OOpH2sWugH9M/7wvJGDenpgHff+byqo2/tXzBwnnOcL79XciFy+ED4gv12Z/okp/sleayxS68/52gbuEy8tGYHLRDeOijVWx6xgB', thinking='The user wants me to write a touching prose poem in Chinese.', type='thinking'), TextBlock(citations=None, text='# 《你走后，灯还亮着》\n\n---\n\n你走的那天，窗外下着细雨。\n\n不是倾盆的那种，就是那种淅淅沥沥、说不清楚的雨。像一句话说了一半，剩下的藏在喉咙里，再也咽不下去，也再也说不出口。\n\n我站在门口送你。你回头看了我一眼，笑了笑，说"进去吧，外面冷。"\n\n我就真的进去了。\n\n后来我想，我应该再多看你一眼的。应该多站一会儿，哪怕雨把我淋透，哪怕什么话都不说，就那样看着你走远，看到你的背影彻底消失在那条老街的转角处。\n\n可我没有。我就那样关上了门。\n\n---\n\n你喜欢在深夜喝茶。不是什么名贵的茶，就是超市里几块钱一包的茉莉花茶。你说香味淡一点，才不会抢了夜晚的安静。\n\n现在每次我路过茶叶铺，那股淡淡的香气一漫出来，我就不得不停下脚步。\n\n不是伤心。就是有那么一瞬间，时间突然变得很慢很慢，慢到我以为你还在某个地方，正捧着一杯热茶，等我回去。\n\n---\n\n我后来搬了家，换了城市，换了天气，换了所有能换的东